## Clinical Correlation — Cortical + Subcortical Volume (Desikan-Killiany + Aseg)

### Overview

This notebook tests whether single-subject **colocalization scores** — derived from the NiSpace
single-subject group comparison pipeline using the **cortical + subcortical volume** parcellation
(Desikan-Killiany atlas cortex + FreeSurfer aseg subcortical structures) — correlate with
clinical severity in Parkinson's disease.

**Logic:** Each subject has one colocalization score per neurotransmitter map per contrast.
That score quantifies how much the subject's deviation from the healthy reference brain spatially
overlaps with a given neurotransmitter receptor/transporter map. Including subcortical structures
(striatum, thalamus, hippocampus, amygdala, etc.) is particularly relevant for PD, where
subcortical atrophy can precede or parallel cortical changes.

**Covariates for residualization:** `age`, `SEX`, `eTIV`  
**Clinical variables:** `moca` (cognition), `gds` (depression), `updrs3_score` (motor severity)  
**Contrasts:** De Novo PD vs HC · Prodromal PD vs HC · De Novo PD vs Prodromal PD

---
**Pipeline overview:**
1. Load cortical+subcortical volume colocalization data, merge in eTIV from the main dataset
2. Residualize clinical variables on age, SEX, eTIV (remove nuisance variance)
3. Z-score the residualized variables
4. Compute Pearson and Spearman correlations per contrast × map × clinical variable
5. FDR-correct p-values within each contrast
6. Visualise as heatmaps; inspect significant associations with scatter plots
7. Export results to CSV

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy.stats import pearsonr, spearmanr, zscore
from statsmodels.stats.multitest import multipletests

### 1. Load data

The merged dataset (`merged_df_volume_desikanaseg.csv`) was produced by the NiSpace pipeline and
contains one row per subject × neurotransmitter map × contrast. Each row includes the
colocalization score and the subject's clinical variables at the matched visit. The parcellation
covers both cortical regions (Desikan-Killiany atlas) and subcortical structures from the
FreeSurfer aseg segmentation (e.g. caudate, putamen, thalamus, hippocampus, amygdala).

**eTIV (estimated Total Intracranial Volume)** is not included in the merged file by default —
it lives in `df1.csv` alongside the FreeSurfer morphometry output. We join it here by `PATNO`.

**Why eTIV instead of field strength?**  
For *thickness* analyses, field strength is the dominant scanner-related nuisance variable
because T1-weighted contrast changes with field strength. For *volume* analyses, however,
individual head size (captured by eTIV) is an additional, continuous source of variance:
larger heads have proportionally larger regional volumes regardless of pathology. Regressing
out eTIV is standard practice in VBM / volumetric studies and removes this scaling effect
before testing clinical associations. This is especially important for subcortical structures,
which scale strongly with total brain size.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR     = Path("../../data")
RESULTS_PATH = Path("../../results")

# ── Load merged dataset (cortical + subcortical volume colocalization scores) ──
df = pd.read_csv(DATA_DIR / "merged_df_volume_desikanaseg.csv", low_memory=False)

# ── Load eTIV and field_strength from df1, merge onto df by PATNO ─────────────
df1      = pd.read_csv(DATA_DIR / "df1.csv", low_memory=False)
etiv_map = df1[["PATNO", "eTIV"]].drop_duplicates("PATNO").set_index("PATNO")["eTIV"]
df["eTIV"] = df["PATNO"].map(etiv_map)

fs_map = df1[["PATNO", "Field Strength"]].drop_duplicates("PATNO").set_index("PATNO")["Field Strength"]
df["field_strength"] = pd.to_numeric(df["PATNO"].map(fs_map), errors="coerce")
df["field_strength"] = df["field_strength"].map({1.5: 0, 3.0: 1})

print(f"Dataset shape: {df.shape}")
print(f"eTIV missing: {df['eTIV'].isna().sum()}")
print(f"field_strength missing: {df['field_strength'].isna().sum()}")
df.head(3)

In [ ]:
# ── Settings ──────────────────────────────────────────────────────────────────
clinical_vars = ["moca", "gds", "updrs3_score"]

covariates = ["age", "SEX", "eTIV", "field_strength"]

### 2. Clean map names

The raw map names in the dataset encode detailed provenance
(target, tracer, sample size, diagnosis, publication). For visualisation we collapse each map
to its neurotransmitter system and primary target, e.g.:

```
Glutamate | target-mGluR5_tracer-abp688_n-73_dx-hc_pub-smart2019
→ Glutamate | mGluR5
```

In [ ]:
def clean_map_name(s):
    try:
        system = s.split("|")[0].strip()
        target = s.split("target-")[1].split("_")[0]
        return f"{system} | {target}"
    except Exception:
        return s

df["map_clean"] = df["map"].apply(clean_map_name)
print("Unique maps after cleaning:")
print(sorted(df["map_clean"].unique()))

### 3. Residualize clinical variables

Before correlating with colocalization scores, we remove the variance in clinical scores
that can be explained by the nuisance covariates `age`, `SEX`, and `eTIV`.

**Method:** Ordinary Least Squares (OLS) regression of each clinical score on the covariates.
The **residuals** are what remains after accounting for those effects — i.e., the
covariate-adjusted clinical severity.

**Why residualize rather than including covariates in the correlation?**  
Residualization is a common two-step approach that cleanly separates the nuisance-removal step
from the main correlation step. It makes subsequent z-scoring and visualisation straightforward,
and the resulting z-scores are directly interpretable as standardised adjusted severity.

**Implementation note:** Rows with missing values in any covariate or the clinical variable are
dropped for that specific regression (listwise deletion). Residuals are then re-indexed back
to the full dataframe, so rows that were excluded remain `NaN` and are handled by
`nan_policy="omit"` during z-scoring.

In [ ]:
def residualize(df, y, covariates):
    """Return OLS residuals of y regressed on covariates (listwise deletion)."""
    sub = df[[y] + covariates].dropna()
    X   = sm.add_constant(sub[covariates])
    model = sm.OLS(sub[y], X).fit()
    # Re-index so that rows dropped due to NaN remain NaN in the output
    resid = pd.Series(np.nan, index=df.index)
    resid.loc[sub.index] = model.resid.values
    return resid

for var in clinical_vars:
    df[f"{var}_adj"] = residualize(df, var, covariates)

# Sanity check: residuals should have mean ≈ 0
print("Residual means (should be ≈ 0):")
for var in clinical_vars:
    m = df[f"{var}_adj"].mean()
    print(f"  {var}_adj: {m:.6f}")

### 4. Z-score the residualized clinical variables

Z-scoring (mean = 0, SD = 1) puts all three clinical variables on a common scale, making
correlation coefficients comparable across variables in the heatmap. The z-scores are computed
across the entire dataset (pooling all subjects and contrasts), which is appropriate here because
the correlation analysis is run separately within each contrast — the z-scores are used only as
the dependent variable in those regressions, not to compare across contrasts directly.

`nan_policy="omit"` ensures that missing values are excluded from the mean/SD calculation
without propagating NaN to other rows.

In [ ]:
for var in clinical_vars:
    df[f"{var}_z"] = zscore(df[f"{var}_adj"], nan_policy="omit")

print("Z-scored variables added:")
print([c for c in df.columns if "_z" in c])

### 5. Correlation analysis

For each combination of **contrast × neurotransmitter map × clinical variable** we compute:

| Statistic | Description |
|---|---|
| **Pearson r** | Parametric correlation; assumes linearity and normality of residuals. Included for completeness and to check agreement with Spearman. |
| **Spearman ρ** | Non-parametric rank correlation; robust to outliers and non-normality. **Primary result.** |

Pairs with fewer than 3 valid observations are skipped (assigned `NaN`).

The nested loop iterates: contrast → map → clinical variable, producing one row per
combination. The `contrast` and `map` levels partition the subjects, so each subset is
independent.

In [ ]:
clinical_z_vars = [f"{v}_z" for v in clinical_vars]
results = []

for contrast in df["contrast"].unique():
    df_c = df[df["contrast"] == contrast]

    for map_name in df_c["map_clean"].unique():
        sub = df_c[df_c["map_clean"] == map_name]

        for var in clinical_z_vars:
            x = sub["colocalization"]
            y = sub[var]
            valid = ~(x.isna() | y.isna())

            if valid.sum() > 2:
                r_p, p_p = pearsonr(x[valid], y[valid])
                r_s, p_s = spearmanr(x[valid], y[valid])
            else:
                r_p = p_p = r_s = p_s = np.nan

            results.append({
                "contrast":     contrast,
                "map":          map_name,
                "clinical_var": var,
                "pearson_r":    r_p,
                "pearson_p":    p_p,
                "spearman_r":   r_s,
                "spearman_p":   p_s,
                "n":            valid.sum(),
            })

results_df = pd.DataFrame(results)
print(f"Total tests: {len(results_df)}")
results_df.head()

### 6. Multiple-comparisons correction (FDR)

We test many map × clinical variable combinations within each contrast, so uncorrected p-values
would be inflated by chance. We apply the **Benjamini-Hochberg False Discovery Rate (FDR)**
procedure independently within each contrast.

**Why FDR rather than Bonferroni?**  
Bonferroni controls the family-wise error rate (probability of *any* false positive) and is
very conservative when tests are positively correlated — which they are here, since
neurotransmitter maps within the same system co-vary. FDR controls the *expected proportion*
of false discoveries among significant findings, offering more power while remaining valid.

**Why correct within contrast, not globally?**  
Each contrast defines a different set of subjects (different patient populations being compared).
The hypotheses tested within each contrast form a natural family; correcting within contrast
avoids penalising tests from unrelated contrasts.

In [ ]:
results_df["pearson_p_fdr"]  = np.nan
results_df["spearman_p_fdr"] = np.nan

for contrast in results_df["contrast"].unique():
    mask_p = (results_df["contrast"] == contrast) & results_df["pearson_p"].notna()
    results_df.loc[mask_p, "pearson_p_fdr"] = multipletests(
        results_df.loc[mask_p, "pearson_p"], method="fdr_bh"
    )[1]

    mask_s = (results_df["contrast"] == contrast) & results_df["spearman_p"].notna()
    results_df.loc[mask_s, "spearman_p_fdr"] = multipletests(
        results_df.loc[mask_s, "spearman_p"], method="fdr_bh"
    )[1]

print("FDR correction done.")
print(f"Significant at FDR < 0.05 (Spearman): {(results_df['spearman_p_fdr'] < 0.05).sum()}")

### 7. Heatmaps

One heatmap per contrast. Rows = neurotransmitter maps, columns = clinical variables.
Cell colour encodes Spearman ρ (diverging blue–red scale, centred at 0).

Significance stars overlay cells that survive FDR correction:  
`*` q < 0.05 · `**` q < 0.01 · `***` q < 0.001

The colour scale is fixed at ±0.15 across all plots so that effect sizes are visually
comparable across contrasts.

In [ ]:
for contrast in results_df["contrast"].unique():
    sub = results_df[results_df["contrast"] == contrast]

    pivot_r = sub.pivot(index="map", columns="clinical_var", values="spearman_r")
    pivot_p = sub.pivot(index="map", columns="clinical_var", values="spearman_p_fdr")

    # Clean column labels for display
    pivot_r.columns = [c.replace("_z", "").replace("_", " ") for c in pivot_r.columns]
    pivot_p.columns = pivot_r.columns

    fig, ax = plt.subplots(figsize=(6, 10))
    sns.heatmap(
        pivot_r,
        ax=ax,
        cmap="bwr",
        center=0,
        vmin=-0.15,
        vmax=0.15,
        annot=True,
        fmt=".2f",
        linewidths=0.5,
        cbar_kws={"label": "Spearman ρ"},
    )

    for i in range(pivot_r.shape[0]):
        for j in range(pivot_r.shape[1]):
            p = pivot_p.iloc[i, j]
            if pd.notna(p):
                if p < 0.001:
                    star = "***"
                elif p < 0.01:
                    star = "**"
                elif p < 0.05:
                    star = "*"
                else:
                    star = ""
                if star:
                    ax.text(j + 0.5, i + 0.75, star,
                            ha="center", va="center", color="black", fontsize=10)

    ax.set_title(
        f"{contrast}\nNeurotransmitter Maps vs Clinical Variables\n"
        "(Spearman ρ, FDR corrected; covariates: age, SEX, eTIV)",
        fontsize=11,
    )
    ax.set_ylabel("Neurotransmitter Map")
    ax.set_xlabel("Clinical Variable")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

### 8. Significant results

Table of all associations that survive FDR correction at q < 0.05, sorted by FDR-adjusted
Spearman p-value. Inspect both the effect size (ρ) and the sample size (n) to judge
practical relevance alongside statistical significance.

In [ ]:
sig = results_df[results_df["spearman_p_fdr"] < 0.05].sort_values("spearman_p_fdr")
print(f"Significant associations: {len(sig)}")
display(sig)

### 9. Scatter plots (sanity check for significant associations)

For each significant association we plot colocalization score (x) against the z-scored
adjusted clinical variable (y), with an OLS regression line.

These serve as a visual sanity check: we verify that the association is not driven by a small
cluster of outliers, that the scatter is roughly linear, and that the direction of the
regression line matches the sign of ρ reported above.

In [ ]:
for _, row in sig.iterrows():
    contrast = row["contrast"]
    map_name = row["map"]
    var      = row["clinical_var"]

    sub = df[(df["contrast"] == contrast) & (df["map_clean"] == map_name)]

    fig, ax = plt.subplots(figsize=(5, 4))
    sns.regplot(x="colocalization", y=var, data=sub, scatter_kws={"alpha": 0.5}, ax=ax)
    ax.set_title(
        f"{contrast}\n{map_name} vs {var}\n"
        f"ρ = {row['spearman_r']:.3f}, FDR-p = {row['spearman_p_fdr']:.4f}"
    )
    ax.set_xlabel("Colocalization score")
    ax.set_ylabel(var)
    plt.tight_layout()
    plt.show()

### 10. Sanity checks

1. **Independence check** — each subject should contribute exactly one row per map × contrast
   combination. If `max > 1`, the dataset has duplicate rows that would inflate correlation
   statistics by artificially increasing N.

2. **Sample sizes** — verify that subject counts per contrast and map are as expected.
   Unexpected drop in N may indicate failed eTIV merges or unexpected missing values.

3. **Colocalization score distribution** — should be roughly unimodal and not heavily skewed.
   Extreme skew or bimodality may signal a data-quality issue and would favour Spearman over
   Pearson (already our primary metric).

4. **Z-scored clinical variable distributions** — after residualization and z-scoring,
   distributions should be approximately symmetric. Large deviations suggest the OLS assumptions
   were violated or that influential outliers are present.

5. **Pearson vs Spearman agreement** — points close to the identity line indicate that the
   relationships are roughly linear and that neither metric is strongly distorted by outliers.
   Systematic divergence would suggest heavy-tailed distributions and would further motivate
   relying on Spearman ρ as the primary statistic.

In [ ]:
# 1. Independence check
print("Max rows per PATNO × map × contrast (should be 1):",
      df.groupby(["PATNO", "map_clean", "contrast"]).size().max())

# 2. Sample sizes
print("\nSample sizes per contrast × map:")
display(results_df.groupby(["contrast", "map"])["n"].first())

# 3. Colocalization score distribution
sns.histplot(df["colocalization"], kde=True)
plt.title("Colocalization score distribution")
plt.show()

# 4. Z-scored clinical variable distributions
for var in clinical_z_vars:
    sns.histplot(df[var].dropna(), kde=True)
    plt.title(f"{var} distribution (z-scored, covariate-adjusted)")
    plt.show()

# 5. Pearson vs Spearman
sns.scatterplot(x=results_df["pearson_r"], y=results_df["spearman_r"])
plt.axline((0, 0), (1, 1), linestyle="--", color="grey")
plt.xlabel("Pearson r")
plt.ylabel("Spearman ρ")
plt.title("Pearson vs Spearman (all tests)")
plt.show()

### 11. Export results

The full results table (all contrasts, maps, clinical variables, both correlation metrics,
raw and FDR-corrected p-values) is saved to CSV for use in downstream reporting or figure
generation.

In [ ]:
RESULTS_PATH.mkdir(parents=True, exist_ok=True)
out = RESULTS_PATH / "clinical_correlation_volume_desikanaseg.csv"
results_df.to_csv(out, index=False)
print(f"Saved: {out}")

---
## Notes

### Statistical approach
- **Residualization (OLS):** Clinical scores are adjusted for `age`, `SEX`, and `eTIV` by
  regressing on these covariates and retaining the residuals. This removes shared variance with
  demographic and head-size factors before the main correlation step.
- **Z-scoring:** Residualized scores are standardised (mean = 0, SD = 1) across the full
  dataset to make effect sizes comparable across clinical variables.
- **Spearman ρ (primary):** Non-parametric rank correlation. Preferred over Pearson because
  clinical scales (UPDRS-III, MoCA, GDS) are ordinal and their distributions are often skewed.
- **Pearson r (secondary):** Shown alongside Spearman for reference; strong agreement between
  the two indicates the relationships are approximately linear.
- **FDR correction (Benjamini-Hochberg):** Applied within each contrast to control the
  expected proportion of false positives. More powerful than Bonferroni when tests are
  correlated, as they are here across neurotransmitter maps from the same system.

### Choice of eTIV over field strength
For cortical *thickness*, field strength is the primary scanner-related nuisance because
T1-weighted contrast scales with B0. For cortical *volume* (and especially subcortical volume),
brain size (eTIV) is an additional and often dominant source of between-subject variance.
Regressing out eTIV is standard practice in VBM studies; it ensures that volumetric
colocalization scores reflect relative atrophy patterns rather than overall brain size.
This is particularly relevant here because subcortical structures (e.g. striatum, thalamus)
scale strongly with total intracranial volume.

### Interpretation
A significant positive ρ between a map and a clinical severity score means: subjects whose
cortical + subcortical volume deviations *spatially overlap more* with that neurotransmitter
system show *more severe* symptoms. A negative ρ means the opposite — greater overlap is
associated with milder symptoms (or protective direction).

Compared to the cortical-only analysis, this parcellation is more sensitive to subcortical
neurodegeneration (e.g. striatal dopaminergic loss in PD) and may therefore show stronger
associations with motor severity (UPDRS-III) or dopaminergic maps.

**Group codes:** CONCOHORT 1 = De Novo PD · 2 = HC · 4 = Prodromal PD